# 02 · Phishing Guard and Coach

A two-agent assistant on an employee's Outlook inbox at Harbor Foods, a fictional food distributor. **Guard** investigates each new email with tools, records a verdict, and asks the employee before moving anything. **Coach** (added Monday) teaches the employee when they call it wrong.

All logic lives in the `phishing_guard` package. Each section below imports the piece it needs and shows it working. Sections follow PLAN.md: Setup, World, Classifier, Tools, Agents, Runners, Demo, Evaluation, Results, Findings.

**Status: Sunday, Oct 4.** Sections 1 to 6 are built. Guard has the five investigate/decide tools; the approval tool, the Coach, and the follow-up tools arrive Monday.

## 1. Setup

The OpenAI key is read from the environment via python-dotenv (`.env` is never printed). If it is missing, `ensure_openai_key()` asks with a hidden prompt. The model is `gpt-5.4-mini`, as in the class tutorial, and tracing is off.

In [ ]:
import json
from phishing_guard.config import ensure_openai_key, has_openai_key, lm_model

ensure_openai_key()
print("model:", lm_model, "| key loaded:", has_openai_key())

## 2. World

Five Python structures hold all state: `USERS`, `MAILBOX`, `THREAT_INTEL`, `TRAINING_LOG`, `SECURITY_QUEUE`. `reset_world()` restores them before every run so each case starts clean. `TEST_CASES` is the answer key; no tool reads it. Mailbox and threat intel load from `data/*.json`.

In [ ]:
from phishing_guard.world import (USERS, MAILBOX, THREAT_INTEL, TRAINING_LOG,
                                  SECURITY_QUEUE, TEST_CASES, reset_world)

reset_world()
print(f"{len(USERS)} users, {len(MAILBOX)} emails, "
      f"{len(THREAT_INTEL['domains'])} domain rows, {len(THREAT_INTEL['urls'])} url rows, "
      f"{len(TEST_CASES)} test cases")
for uid, u in USERS.items():
    print(f"  {uid} {u['name']:<12} {u['role']:<18} knows {u['known_domains']}")

## 3. Classifier

Loads `phish_clf.joblib` trained in notebook 01. `top_words` lists the words that pushed an email's score toward phishing, which Guard sees as part of the tool result. Below, the classifier alone on the ten development emails: it should flag the obvious ones and miss the social-engineering ones that have no classic phishing vocabulary.

In [ ]:
from phishing_guard.classifier import load_classifier, top_words

clf = load_classifier()
truth = {c['email_id']: c['true_label'] for c in TEST_CASES.values()}
print(f"{'email':<7}{'p(phish)':<10}{'truth':<12}top words")
for eid, e in sorted(MAILBOX.items()):
    text = e['subject'] + ' ' + e['body']
    p = float(clf.predict_proba([text])[0][1])
    print(f"{eid:<7}{p:<10.2f}{truth[eid]:<12}{top_words(text)}")

## 4. Tools

Guard's tools are `@function_tool` functions in `phishing_guard/tools.py`. Four gather evidence and one records the verdict. `check_sender_domain` and `check_url` take the value itself, so Guard has to extract the right domain or link, which is what the argument-correctness metric checks. Every tool returns data on failure and never raises.

Monday adds `move_to_phishing_folder` (human approval plus a guardrail), `log_training_event`, and `escalate_to_security`.

In [ ]:
from phishing_guard.tools import SUNDAY_TOOLS

for t in SUNDAY_TOOLS:
    params = list(t.params_json_schema['properties'])
    print(f"{t.name:<22} {params}")

## 5. Agents

Guard's instructions follow five labeled stages (INVESTIGATE, DECIDE, ACT, FOLLOW UP, REPLY) so the trace can be checked against them. `build_input` turns a mailbox record into the message Guard receives; it leaves out `is_simulation` and every label.

On Sunday the ACT and FOLLOW UP tools do not exist yet, so a short addendum tells Guard to stop after the verdict. Coach arrives Monday.

In [ ]:
from phishing_guard.guard_agents import make_guard, build_input, SUNDAY_ADDENDUM

guard = make_guard(SUNDAY_TOOLS, SUNDAY_ADDENDUM)
print("Guard tools:", [t.name for t in guard.tools])
print()
print(build_input("E-002"))

## 6. Runners

`show_trace` is the tutorial's helper for reading `result.new_items`. `run_investigation` resets the world and runs Guard on one email. Monday adds `run_interactive` (a person approves or rejects) and `run_case` (the decision comes from the test case).

**Sunday end-of-day check.** Guard should record a sensible verdict on demo cases A, B, and C. Each run costs a few model calls.

In [ ]:
from agents import Runner
from phishing_guard.trace_utils import show_trace
from phishing_guard.runners import run_investigation, recorded_verdict

expected = {c['email_id']: c['expected_verdict'] for c in TEST_CASES.values()}
summary = []
for eid in ["E-001", "E-002", "E-003"]:
    print("=" * 72, f"\n{eid}  expected: {expected[eid]}\n" + "=" * 72)
    result = await run_investigation(guard, eid)
    show_trace(result)
    rv = recorded_verdict(eid)
    print("\nRecorded:", rv)
    summary.append((eid, expected[eid], rv['verdict']))

print("\nSUMMARY")
for eid, exp, rec in summary:
    print(f"{eid}  expected={exp:<12} recorded={rec}")

## 7. Demo · 8. Evaluation · 9. Results · 10. Findings

Built Monday and Tuesday per PLAN.md.